## parameterised profile modelling

#### Generate an I beam using IFC

The I beam profile shape is pre-defined in IFC, along with a set of other profiles. Therefore the actual geometry isnt pre-defined in terms of primitives, but abstracted away by a set of parameters. However there are instructions in IFCopenShell to calculate each of these profiles using a set of control points.


In [ ]:
import ifcopenshell
import ifcopenshell.api

# 1. Create a blank IFC model
model = ifcopenshell.file(schema="IFC4")

# 2. Setup Project Hierarchy
project = ifcopenshell.api.run("root.create_entity", model, ifc_class="IfcProject", name="Demo Project")

# --- FIX IS HERE ---
# We assign units once. We specify "MILLIMETERS" for length. 
# The API automatically handles the conversion to scientific notation (exponent -3).
ifcopenshell.api.run("unit.assign_unit", model, length={"is_metric": True, "exponent": -3, "raw": "MILLIMETERS"})# -------------------

site = ifcopenshell.api.run("root.create_entity", model, ifc_class="IfcSite", name="My Site")
building = ifcopenshell.api.run("root.create_entity", model, ifc_class="IfcBuilding", name="My Building")
storey = ifcopenshell.api.run("root.create_entity", model, ifc_class="IfcBuildingStorey", name="Ground Floor")

# Link hierarchy
ifcopenshell.api.run("aggregate.assign_object", model, relating_object=project, product=site)
ifcopenshell.api.run("aggregate.assign_object", model, relating_object=site, product=building)
ifcopenshell.api.run("aggregate.assign_object", model, relating_object=building, product=storey)

# 3. Create the Context
model_context = ifcopenshell.api.run("context.add_context", model, context_type="Model")
body_context = ifcopenshell.api.run("context.add_context", model, 
    context_type="Model", context_identifier="Body", target_view="MODEL_VIEW", parent=model_context)

# 4. Create the IfcIShapeProfileDef (HEA 300)
# Measurements are in MILLIMETERS now
profile = model.create_entity("IfcIShapeProfileDef",
    ProfileName="HEA300",
    ProfileType="AREA",
    OverallWidth=300.0,
    OverallDepth=290.0,
    WebThickness=8.5,
    FlangeThickness=14.0,
    FilletRadius=27.0
)

# 5. Create the Beam Element
beam = ifcopenshell.api.run("root.create_entity", model, ifc_class="IfcBeam", name="Steel I-Beam")

# 6. Assign Geometry (Extrude 5000mm)
representation = ifcopenshell.api.run("geometry.add_profile_representation", model, 
    context=body_context, 
    profile=profile, 
    depth=5.0
)
ifcopenshell.api.run("geometry.assign_representation", model, product=beam, representation=representation)

# 7. Place the Beam
ifcopenshell.api.run("spatial.assign_container", model, relating_structure=storey, product=beam)
ifcopenshell.api.run("geometry.edit_object_placement", model, product=beam, matrix=[[1,0,0,0], [0,1,0,0], [0,0,1,0], [0,0,0,1]])

# 8. Save
filename = "simple_i_beam.ifc"
model.write(filename)
print(f"Successfully created {filename}")

#### Recreate profile programmatically and sample a set of points along the profile

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

class IBeamSection:
    def __init__(self, width, depth, web_thickness, flange_thickness, fillet_radius=0):
        self.x1 = width / 2.0
        self.y = depth / 2.0
        self.d1 = web_thickness / 2.0
        self.ft1 = flange_thickness
        self.f1 = fillet_radius
        
        self.segments = [] 
        self._build_geometry()

    def _build_geometry(self):
        # --- Quadrant 1: Bottom Right ---
        self.segments.append(Line((-self.x1, -self.y), (self.x1, -self.y)))
        self.segments.append(Line((self.x1, -self.y), (self.x1, -self.y + self.ft1)))
        
        # Bottom Right Fillet
        start_x = self.x1
        end_x = self.d1 + self.f1
        y_level = -self.y + self.ft1
        
        if end_x < start_x:
            self.segments.append(Line((start_x, y_level), (end_x, y_level)))
        
        if self.f1 > 0:
            center = (self.d1 + self.f1, -self.y + self.ft1 + self.f1)
            # FIX: Go from -90 to -180 (Clockwise/Negative direction)
            # This ensures a simple 90 degree turn instead of the long way around
            self.segments.append(Arc(center, self.f1, start_angle=-90, end_angle=-180))

        # --- Quadrant 2: Top Right ---
        start_y = -self.y + self.ft1 + self.f1
        end_y = self.y - self.ft1 - self.f1
        
        if end_y > start_y:
            self.segments.append(Line((self.d1, start_y), (self.d1, end_y)))
            
        if self.f1 > 0:
            center = (self.d1 + self.f1, self.y - self.ft1 - self.f1)
            # From 180 (left) to 90 (up)
            self.segments.append(Arc(center, self.f1, start_angle=180, end_angle=90))
            
        start_x = self.d1 + self.f1
        end_x = self.x1
        y_level = self.y - self.ft1
        self.segments.append(Line((start_x, y_level), (end_x, y_level)))
        
        self.segments.append(Line((self.x1, self.y - self.ft1), (self.x1, self.y)))
        self.segments.append(Line((self.x1, self.y), (-self.x1, self.y)))
        
        # --- Quadrant 3: Top Left ---
        self.segments.append(Line((-self.x1, self.y), (-self.x1, self.y - self.ft1)))
        self.segments.append(Line((-self.x1, self.y - self.ft1), (-self.d1 - self.f1, self.y - self.ft1)))
        
        if self.f1 > 0:
            center = (-self.d1 - self.f1, self.y - self.ft1 - self.f1)
            # From 90 (up) to 0 (right)
            self.segments.append(Arc(center, self.f1, start_angle=90, end_angle=0))

        start_y = self.y - self.ft1 - self.f1
        end_y = -self.y + self.ft1 + self.f1
        self.segments.append(Line((-self.d1, start_y), (-self.d1, end_y)))
        
        # --- Quadrant 4: Bottom Left ---
        if self.f1 > 0:
            center = (-self.d1 - self.f1, -self.y + self.ft1 + self.f1)
            # From 360 (Right) to 270 (Bottom)
            self.segments.append(Arc(center, self.f1, start_angle=360, end_angle=270))

        self.segments.append(Line((-self.d1 - self.f1, -self.y + self.ft1), (-self.x1, -self.y + self.ft1)))
        self.segments.append(Line((-self.x1, -self.y + self.ft1), (-self.x1, -self.y)))

    def get_sample_points(self, n):
        total_length = sum(s.length for s in self.segments)
        step_size = total_length / (n - 1)
        
        points = []
        current_segment_idx = 0
        dist_covered_in_prev_segments = 0.0
        
        for i in range(n):
            target_dist = i * step_size
            
            # Walk forward until we find the segment
            while True:
                segment = self.segments[current_segment_idx]
                segment_end_dist = dist_covered_in_prev_segments + segment.length
                
                # Small tolerance to catch floating point errors at the exact end
                if target_dist <= segment_end_dist + 1e-9:
                    local_dist = target_dist - dist_covered_in_prev_segments
                    points.append(segment.get_point_at_dist(local_dist))
                    break
                else:
                    dist_covered_in_prev_segments += segment.length
                    current_segment_idx += 1
                    if current_segment_idx >= len(self.segments):
                        points.append(self.segments[-1].end)
                        break
        return points

    def get_3d_points(self, length, center, axis, n_longitudinal=20, n_profile=50):
        """
        Generates 3D points for the I-beam with progressive offset sampling.
        
        :param length: Length of the beam
        :param center: 3D center point of the beam (x, y, z)
        :param axis: 3D vector representing the longitudinal axis of the beam
        :param n_longitudinal: Number of cross-sections to sample along the length
        :param n_profile: Number of points to sample per cross-section
        :return: numpy array of shape (N, 3) containing the 3D points
        """
        # 1. Generate a high-resolution profile (10x more points)
        high_res_profile = self.get_sample_points(n_profile * 10)
        
        # 2. Define longitudinal steps (local z)
        # Centered around 0: from -length/2 to length/2
        z_steps = np.linspace(-length/2, length/2, n_longitudinal)
        
        points_3d = []
        
        # 3. Sample with progressive offset
        # For each slice, we start at a different offset and pick every 10th point
        for i, z in enumerate(z_steps):
            # Start offset increases by 1 for each slice
            start_offset = i % 10
            
            # Pick n_profile points starting from start_offset, every 10th point
            for j in range(n_profile):
                idx = start_offset + j * 10
                p2 = high_res_profile[idx]
                points_3d.append([p2[0], p2[1], z])
                    
        points_3d = np.array(points_3d)
        
        # 4. Compute Rotation Matrix to align Local Z (0,0,1) with Target Axis
        target_axis = np.array(axis, dtype=float)
        norm = np.linalg.norm(target_axis)
        if norm == 0:
            raise ValueError("Axis vector cannot be zero.")
        target_axis = target_axis / norm
        
        local_z = np.array([0, 0, 1])
        
        if np.allclose(target_axis, local_z):
            R = np.eye(3)
        elif np.allclose(target_axis, -local_z):
            # 180 degree rotation around X
            R = np.array([[1, 0, 0], [0, -1, 0], [0, 0, -1]])
        else:
            # Construct an orthonormal basis {u, v, w} where w = target_axis
            # We need a stable 'up' vector to define the orientation of the cross-section
            # Arbitrarily choose X or Y axis as reference
            if np.abs(target_axis[2]) < 0.9:
                ref_vector = np.array([0, 0, 1])
            else:
                ref_vector = np.array([1, 0, 0])
                
            u = np.cross(ref_vector, target_axis)
            u = u / np.linalg.norm(u)
            
            v = np.cross(target_axis, u)
            v = v / np.linalg.norm(v)
            
            # R = [u, v, target_axis] maps [1,0,0]->u, [0,1,0]->v, [0,0,1]->target_axis
            R = np.column_stack((u, v, target_axis))
            
        # 5. Apply Rotation and Translation
        # P_global = Center + R @ P_local
        center = np.array(center)
        
        # points_3d is (N, 3). We want (R @ points_3d.T).T + center
        transformed_points = (points_3d @ R.T) + center
        
        return transformed_points

class Line:
    def __init__(self, start, end):
        self.start = np.array(start)
        self.end = np.array(end)
        self.vec = self.end - self.start
        self.length = np.linalg.norm(self.vec)
        
    def get_point_at_dist(self, d):
        if self.length == 0: return self.start
        # Clamp d to [0, length] to avoid slight overshoots
        d = max(0, min(d, self.length))
        t = d / self.length
        return self.start + (self.vec * t)

class Arc:
    def __init__(self, center, radius, start_angle, end_angle):
        self.center = np.array(center)
        self.radius = radius
        self.start_rad = math.radians(start_angle)
        self.end_rad = math.radians(end_angle)
        
        # FIX: Calculate actual angular sweep
        self.angle_diff = self.end_rad - self.start_rad
        self.length = abs(self.angle_diff * radius)

    def get_point_at_dist(self, d):
        d = max(0, min(d, self.length))
        t = d / self.length
        # Interpolate
        angle = self.start_rad + t * self.angle_diff
        x = self.center[0] + self.radius * math.cos(angle)
        y = self.center[1] + self.radius * math.sin(angle)
        return np.array([x, y])

# --- Usage ---
profile = IBeamSection(width=300, depth=290, web_thickness=18.5, flange_thickness=14, fillet_radius=27)

# Define 3D parameters
beam_length = 1000.0
beam_center = [500, 500, 500]
beam_axis = [1, 1, 1]  # Diagonal axis

# Generate 3D points
points_3d = profile.get_3d_points(length=beam_length, center=beam_center, axis=beam_axis, 
                                   n_longitudinal=20, n_profile=50)

# Plotting
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

xs = points_3d[:, 0]
ys = points_3d[:, 1]
zs = points_3d[:, 2]

ax.scatter(xs, ys, zs, s=1, c=zs, cmap='viridis')

# Set equal aspect ratio for 3D plot
max_range = np.array([xs.max()-xs.min(), ys.max()-ys.min(), zs.max()-zs.min()]).max() / 2.0
mid_x = (xs.max()+xs.min()) * 0.5
mid_y = (ys.max()+ys.min()) * 0.5
mid_z = (zs.max()+zs.min()) * 0.5
ax.set_xlim(mid_x - max_range, mid_x + max_range)
ax.set_ylim(mid_y - max_range, mid_y + max_range)
ax.set_zlim(mid_z - max_range, mid_z + max_range)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.title(f"3D HEA 300 Profile Sampled Points")
plt.show()

# Save as PLY file
def save_ply(points, filename):
    """
    Save point cloud to PLY format.
    
    :param points: numpy array of shape (N, 3)
    :param filename: output filename
    """
    with open(filename, 'w') as f:
        # Write header
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {len(points)}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        f.write("end_header\n")
        
        # Write vertices
        for point in points:
            f.write(f"{point[0]} {point[1]} {point[2]}\n")
    
    print(f"Saved {len(points)} points to {filename}")

# Save the point cloud
save_ply(points_3d, "i_beam_profile.ply")

In [ ]:
## GPU VERSION

import torch
import torch.nn.functional as F


In [ ]:
# Example usage of GPU I-beam generation
import torch
import numpy as np

# Create a batch of I-beam parameters
batch_size = 4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize parameter tensor
# Format: [width, depth, web_thickness, flange_thickness, fillet_radius, length, dir_x, dir_y, dir_z, center_x, center_y, center_z]
preds = torch.zeros(batch_size, 12, device=device)

# Set parameters for each beam in batch
preds[:, 0] = 300.0   # width
preds[:, 1] = 290.0   # depth
preds[:, 2] = 18.5    # web_thickness
preds[:, 3] = 14.0    # flange_thickness
preds[:, 4] = 27.0    # fillet_radius
preds[:, 5] = 1000.0  # length

# Set different directions and centers for each beam
preds[0, 6:9] = torch.tensor([1.0, 0.0, 0.0], device=device)  # X-axis
preds[0, 9:12] = torch.tensor([0.0, 0.0, 0.0], device=device)

preds[1, 6:9] = torch.tensor([0.0, 1.0, 0.0], device=device)  # Y-axis
preds[1, 9:12] = torch.tensor([2000.0, 0.0, 0.0], device=device)

preds[2, 6:9] = torch.tensor([0.0, 0.0, 1.0], device=device)  # Z-axis
preds[2, 9:12] = torch.tensor([4000.0, 0.0, 0.0], device=device)

preds[3, 6:9] = torch.tensor([1.0, 1.0, 1.0], device=device)  # Diagonal
preds[3, 9:12] = torch.tensor([6000.0, 0.0, 0.0], device=device)

# Generate point clouds on GPU
print(f"Generating {batch_size} I-beams on {device}...")
points_gpu = generate_ibeam_cloud_tensor(preds, n_longitudinal=20, n_profile=50)
print(f"Generated tensor shape: {points_gpu.shape}")  # Should be (batch_size, 1000, 3)

# Convert to CPU and numpy
points_cpu = points_gpu.cpu().numpy()
print(f"Converted to numpy array shape: {points_cpu.shape}")

# Save each beam to a separate PLY file
def save_ply_batch(points_array, filename_prefix):
    """
    Save batch of point clouds to PLY files.
    
    :param points_array: numpy array of shape (batch_size, n_points, 3)
    :param filename_prefix: prefix for output filenames
    """
    batch_size = points_array.shape[0]
    
    for i in range(batch_size):
        points = points_array[i]
        filename = f"{filename_prefix}_beam_{i}.ply"
        
        with open(filename, 'w') as f:
            # Write header
            f.write("ply\n")
            f.write("format ascii 1.0\n")
            f.write(f"element vertex {len(points)}\n")
            f.write("property float x\n")
            f.write("property float y\n")
            f.write("property float z\n")
            f.write("end_header\n")
            
            # Write vertices
            for point in points:
                f.write(f"{point[0]} {point[1]} {point[2]}\n")
        
        print(f"Saved beam {i} with {len(points)} points to {filename}")

# Save all beams
save_ply_batch(points_cpu, "i_beam_gpu")

# Optionally: Combine all beams into a single point cloud and save
all_points = points_cpu.reshape(-1, 3)  # Flatten to (batch_size * n_points, 3)
print(f"\nCombined all beams: {all_points.shape[0]} total points")

with open("i_beam_gpu_combined.ply", 'w') as f:
    f.write("ply\n")
    f.write("format ascii 1.0\n")
    f.write(f"element vertex {len(all_points)}\n")
    f.write("property float x\n")
    f.write("property float y\n")
    f.write("property float z\n")
    f.write("end_header\n")
    
    for point in all_points:
        f.write(f"{point[0]} {point[1]} {point[2]}\n")

print(f"Saved combined point cloud to i_beam_gpu_combined.ply")